In [3]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.io as pio

from gensim.corpora import Dictionary
from gensim.models import word2vec

from sklearn.manifold import TSNE as tsne

import plotly.io as pio
pio.renderers.default = "vscode"

In [4]:
OHCO = ['book_title', 'chap_num', 'para_num', 'sent_num', 'token_num']
BAG = OHCO[:2]

In [5]:
TOKENS=pd.read_csv("Corpus_Tokens.csv").set_index(OHCO)

In [6]:
docs = TOKENS.groupby(BAG).term_str.apply(list).tolist()

In [7]:
w2v_params = dict(window = 5,
    vector_size = 100,
    min_count = 250,
    workers = 4
)

In [8]:
model = word2vec.Word2Vec(docs, **w2v_params)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [9]:
WV = pd.DataFrame(model.wv.vectors, index=model.wv.index_to_key)
WV.index.name = 'term_str'
WV

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
term_str,,,,,,,,,,,,,,,,,,,,,
the,-0.331478,-0.323553,0.085222,0.468696,-0.847656,-1.103998,0.867537,0.806901,-0.391110,-0.327434,...,-0.438631,0.168836,0.118338,0.144902,0.709303,0.011741,-0.923670,-0.479885,-0.397411,-0.410102
and,-0.506499,0.672144,-0.066248,0.257934,-0.208727,-1.161461,-0.446157,0.341102,0.337635,-0.109463,...,-0.271133,0.505217,-0.006720,0.070123,-1.020992,-0.036072,0.606981,0.391708,-0.430684,0.358115
to,0.865395,-0.045856,0.198557,-1.445379,1.425930,0.276581,-1.985171,-0.515027,-0.141375,1.606297,...,0.652654,0.127835,0.288212,-0.550503,-1.068692,-0.284679,-0.552019,0.786788,1.903667,-0.134256
of,-0.031162,-0.022257,0.393907,-0.595390,0.043685,1.205987,-0.721191,-0.664938,0.687065,-0.926771,...,0.275317,0.188543,-0.154694,0.184892,-0.166393,-0.137502,0.946231,-0.403550,0.028420,-1.059305
a,-0.888787,-0.297907,-1.006870,0.044074,1.137653,-0.512254,0.491017,1.767888,-1.236208,-0.499063,...,-0.035114,0.742300,0.092307,-0.225806,0.771129,-1.285176,-0.204013,-1.349519,0.658055,0.348149
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
relief,0.470583,-0.440103,0.186748,-0.164839,-0.027892,0.125960,-0.719073,-0.159427,-0.456674,0.124448,...,-0.269429,0.260645,-0.019086,-0.201465,0.086046,-0.159942,-0.253362,-0.460368,0.179300,0.905610
especially,-0.143931,-0.266022,0.036807,-0.611946,-0.199474,-0.445189,0.323606,-0.056461,0.667799,-0.661020,...,0.153991,-0.047226,-0.370025,-0.257555,-0.537728,0.051718,-0.208692,-0.051664,0.457873,0.221280
credit,1.087316,-0.025416,0.150251,-0.513007,0.229392,0.673735,-0.212446,-0.541166,0.252719,0.628225,...,-0.672425,-0.286621,-0.867820,0.081428,0.250848,-0.859211,0.058496,-0.578407,0.633605,-0.285323


In [10]:
PP = 50

tsne_engine = tsne(
    perplexity=PP, 
    n_components=2, 
    init='pca', 
    max_iter=2500, 
    random_state=23
)
TSNE = pd.DataFrame(
    tsne_engine.fit_transform(WV), 
    columns=['x','y'], 
    index=WV.index)
TSNE

,x,y
term_str,,
the,3.243907,-3.041849
and,4.666877,-2.861546
to,-6.078312,9.556374
of,-2.332233,1.613962
a,-1.763030,-1.594896
...,...,...
relief,4.806556,7.625368
especially,-0.915384,2.441158
credit,0.781691,15.180038


In [11]:
fig = px.scatter(
    TSNE.reset_index(),
    x='x',
    y='y',
    text='term_str',
    hover_name='term_str',
    height=1000,
    width=1200
)

fig.update_traces(
    mode='markers+text',
    textfont=dict(color='black', size=14, family='Arial'),
    textposition='top center'
)


fig.update_layout(
    title=dict(
        text="t-SNE Projection of Word2Vec Embeddings",
        x=0.5,     
        xanchor='center',
        font=dict(size=20)
    )
)

fig.update_layout(
    xaxis_title="x",
    yaxis_title="y"
)

fig.show()

## Riff 2

In [12]:
def complete_analogy(A, B, C, n=5):
    try:
        cols = ['term', 'sim']
        return pd.DataFrame(model.wv.most_similar(positive=[B, C], negative=[A])[0:n], columns=cols)
    except KeyError as e:
        print('Error:', e)
        return None
    
def get_most_similar(positive, negative=None):
    return pd.DataFrame(model.wv.most_similar(positive, negative), columns=['term', 'sim'])

In [22]:
complete_analogy("money","power","poor")

,term,sim
0,innocent,0.616465
1,beauty,0.579868
2,sweet,0.548051
3,gentle,0.534283
4,wretched,0.525698


In [48]:
complete_analogy("prisoner","prison","person")

,term,sim
0,town,0.474408
1,court,0.433134
2,house,0.430255
3,church,0.424060
4,being,0.423624


In [29]:
get_most_similar('poor')

,term,sim
0,child,0.635706
1,dearest,0.618941
2,darling,0.541996
3,papa,0.534986
4,dear,0.525086
5,mother,0.523970
6,baby,0.515498
7,sweet,0.500865
8,miserable,0.498908
9,pa,0.497776


In [14]:
WV.to_csv("VOCAB_W2V.csv")